# Sleep-stage CNN classifier

This notebook follows the paper-inspired design: each 30-second target epoch uses five preceding, the target, and four following epochs. The input channels are Polar heart rate (1 Hz), CIN2, and CIN4 (20 Hz), all decimated to `MODEL_RATE` (2 Hz) before entering the network. Apple Watch stages are used only as labels. By default one complete sleep (`VALIDATION_DATE`) is held out for validation; with `CROSS_VALIDATE = True` the model is trained once per sleep with that sleep held out, each fold is reported, and the mean and standard deviation over the folds are printed at the end.

Each branch encodes the 10 context epochs with a shared convolutional encoder (Conv1d, batch normalization, ReLU, max-pooling by 2, global average pooling), mixes the 10 epoch embeddings with a small convolution across epochs, and classifies the center epoch. Training windows are taken with a 10-second stride (validation windows stay non-overlapping), and the loss uses square-root inverse-frequency class weights with label smoothing. Training uses He initialization, AdamW (`lr=1e-3`, weight decay `1e-2`), batch size 100, a factor-2 learning-rate reduction after 10 epochs without validation-loss improvement, and early stopping after 30 epochs without improvement.

`TRAINING_MODE` selects how heart rate and the eye sensor are combined:
- **`"staged"`**: stage 1 trains an HR-only branch. Stage 2 adds a separate CIN branch through a zero-initialized fusion head, so it starts exactly at the HR model; the HR branch is fine-tuned at `HR_FINETUNE_LR` (0 freezes it). Stage 2 is always retained and evaluated as the multimodal model.
- **`"joint"`**: all three channels enter one branch and are trained together. In leave-one-night-out tests this scored higher (mean kappa 0.345 vs 0.316 for staged, 0.301 for HR only).

In [7]:
from __future__ import annotations

import copy
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.signal import butter, resample_poly, sosfiltfilt
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATASET_DIR = Path.cwd()
if DATASET_DIR.name != "datasets_for_training":
    raise RuntimeError("Open and run this notebook with datasets_for_training as the working directory.")
VALIDATION_DATE = "2026-10-17"
CROSS_VALIDATE = True  # True: train once per sleep with that sleep held out, then print the average over all folds
SAMPLE_RATE = 20
MODEL_RATE = 2
FDC_HIGH_PASS_CUTOFF_HZ = 0.1
EPOCH_SECONDS = 30
TRAIN_STRIDE_SECONDS = 10
CONTEXT_BEFORE = 5
CONTEXT_AFTER = 4
CONTEXT_EPOCHS = CONTEXT_BEFORE + 1 + CONTEXT_AFTER
SAMPLES_PER_EPOCH = EPOCH_SECONDS * MODEL_RATE
INPUT_SAMPLES = CONTEXT_EPOCHS * SAMPLES_PER_EPOCH
TRAINING_MODE = "staged"  # "staged": HR branch first, then CIN added; "joint": all channels trained together
HR_FINETUNE_LR = 1e-4  # staged mode: HR-branch learning rate in stage 2 (0 freezes it)
BATCH_SIZE = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-2
DROPOUT = 0.3
ENCODER_WIDTH = 32
LABEL_SMOOTHING = 0.1
CLASS_WEIGHT_POWER = 0.5
MAX_EPOCHS = 300
LR_PATIENCE = 10
EARLY_STOPPING_PATIENCE = 30
STAGES = ["Awake", "REM", "Core", "Deep"]
STAGE_TO_INDEX = {stage: index for index, stage in enumerate(STAGES)}
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required by this notebook, but torch.cuda.is_available() is False.")
DEVICE = torch.device("cuda")
# GPU speed-ups: cuDNN picks the fastest convolution algorithms, and forward/backward passes run in bfloat16
# mixed precision (BatchNorm and the loss stay in float32). Set AUTOCAST_DTYPE = None for full float32.
torch.backends.cudnn.benchmark = True
AUTOCAST_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else None
print("Dataset:", DATASET_DIR)
print("Device:", torch.cuda.get_device_name(DEVICE))
print("PyTorch:", torch.__version__, "| mixed precision:", AUTOCAST_DTYPE)
print("Validation:", "every sleep in turn (cross-validation)" if CROSS_VALIDATE else VALIDATION_DATE)

Dataset: c:\BME\Szemfedo\CapMeter\data_analysis\sleep_tests\datasets_for_training
Device: NVIDIA GeForce RTX 3050 6GB Laptop GPU
PyTorch: 2.11.0+cu128 | mixed precision: torch.bfloat16
Validation: every sleep in turn (cross-validation)


## Loading and aligning the three sources

The trimmed files are treated as synchronized recordings: time zero is the first sample in each trimmed file. FDC timestamps are converted to seconds relative to their first row; Polar time is converted to seconds relative to its first retained sample; Apple Watch intervals are converted relative to its first interval. The common interval is represented at the native 20 Hz FDC rate. The 1 Hz heart-rate signal is linearly interpolated onto that 20 Hz grid.

In [8]:
def _date_from_name(path: Path) -> str:
    match = re.fullmatch(r"(\d{4}-\d{2}-\d{2})", path.stem)
    if match is None:
        raise ValueError(f"Unexpected Apple Watch filename: {path.name}")
    return match.group(1)


def _find_files(recording_date: str) -> tuple[Path, Path, Path]:
    apple = DATASET_DIR / f"{recording_date}.csv"
    fdc = DATASET_DIR / f"fdc1004_data_{recording_date.replace('-', '_')}.csv"
    polar_files = sorted(DATASET_DIR.glob(f"polar_{recording_date}_*.CSV"))
    if not apple.is_file() or not fdc.is_file() or not polar_files:
        raise FileNotFoundError(f"Missing one or more files for {recording_date}")
    return apple, polar_files[0], fdc


def _load_sleep(recording_date: str) -> pd.DataFrame:
    apple_path, polar_path, fdc_path = _find_files(recording_date)
    apple = pd.read_csv(apple_path)
    for column in ("Start time", "End time"):
        apple[column] = pd.to_datetime(apple[column]).map(
            lambda value: value.tz_localize(None) if value.tzinfo else value
        )
    apple_start = apple["Start time"].min()
    apple["start_s"] = (apple["Start time"] - apple_start).dt.total_seconds()
    apple["end_s"] = (apple["End time"] - apple_start).dt.total_seconds()

    polar = pd.read_csv(polar_path)
    polar["time_s"] = pd.to_timedelta(polar["Time"]).dt.total_seconds()
    polar["time_s"] -= polar["time_s"].min()
    polar["HR (bpm)"] = pd.to_numeric(polar["HR (bpm)"], errors="coerce")
    polar = polar.dropna(subset=["time_s", "HR (bpm)"]).drop_duplicates("time_s")

    fdc = pd.read_csv(fdc_path, usecols=["timestamp_ms", "CIN2_pF", "CIN4_pF"])
    timestamp_ms = pd.to_numeric(fdc["timestamp_ms"], errors="coerce")
    fdc["time_s"] = (timestamp_ms - timestamp_ms.iloc[0]) / 1000
    for column in ("CIN2_pF", "CIN4_pF"):
        fdc[column] = pd.to_numeric(fdc[column], errors="coerce")
    fdc = fdc.dropna(subset=["time_s", "CIN2_pF", "CIN4_pF"])
    fdc_filter = butter(
        N=4,
        Wn=FDC_HIGH_PASS_CUTOFF_HZ,
        btype="highpass",
        fs=SAMPLE_RATE,
        output="sos",
    )
    for column in ("CIN2_pF", "CIN4_pF"):
        fdc[column] = sosfiltfilt(fdc_filter, fdc[column].to_numpy(np.float64)).astype(np.float32)

    duration = min(float(apple["end_s"].max()), float(polar["time_s"].max()), float(fdc["time_s"].max()))
    samples = np.arange(0, math.floor(duration * SAMPLE_RATE) + 1, dtype=np.float32)
    seconds = samples / SAMPLE_RATE
    result = pd.DataFrame({"time_s": seconds})
    for column in ("CIN2_pF", "CIN4_pF"):
        result[column] = np.interp(seconds, fdc["time_s"], fdc[column])
    result["HR (bpm)"] = np.interp(seconds, polar["time_s"], polar["HR (bpm)"])
    result["stage"] = pd.NA
    for row in apple.itertuples():
        mask = (result["time_s"] >= row.start_s) & (result["time_s"] < row.end_s)
        result.loc[mask, "stage"] = row._1
    return result.dropna(subset=["stage"]).reset_index(drop=True)


recording_dates = sorted(_date_from_name(path) for path in DATASET_DIR.glob("20??-??-??.csv"))
if len(recording_dates) != 4:
    raise ValueError(f"Expected four training sleeps, found: {recording_dates}")
if VALIDATION_DATE not in recording_dates:
    raise ValueError(f"VALIDATION_DATE={VALIDATION_DATE} is not one of {recording_dates}")
sleep_data = {recording_date: _load_sleep(recording_date) for recording_date in recording_dates}
for recording_date, data in sleep_data.items():
    print(recording_date, len(data), "20-Hz rows", data["stage"].value_counts().to_dict())

C:\Users\Roli\AppData\Local\Temp\ipykernel_18504\1789997422.py:34: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  fdc = pd.read_csv(fdc_path, usecols=["timestamp_ms", "CIN2_pF", "CIN4_pF"])
C:\Users\Roli\AppData\Local\Temp\ipykernel_18504\1789997422.py:34: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  fdc = pd.read_csv(fdc_path, usecols=["timestamp_ms", "CIN2_pF", "CIN4_pF"])


2026-10-16 443081 20-Hz rows {'Core': 234600, 'REM': 107460, 'Deep': 78620, 'Awake': 22401}
2026-10-17 363241 20-Hz rows {'Core': 151021, 'REM': 102980, 'Deep': 85220, 'Awake': 24020}
2026-10-19 333641 20-Hz rows {'Core': 145880, 'REM': 89440, 'Deep': 85820, 'Awake': 12501}
2026-10-20 188699 20-Hz rows {'Core': 107420, 'Deep': 52820, 'REM': 18779, 'Awake': 9680}


## Epoch and context-window construction

The 20 Hz signals are decimated to `MODEL_RATE` with an anti-aliasing polyphase filter. Each label is the majority Apple Watch stage in the center 30-second epoch. Only complete 10-epoch windows are used. Training windows start every `TRAIN_STRIDE_SECONDS`, which roughly triples the number of training examples; validation windows are non-overlapping, so each validation epoch is scored once.

In [9]:
def _make_examples(data: pd.DataFrame, stride_seconds: int = EPOCH_SECONDS) -> tuple[np.ndarray, np.ndarray]:
    factor = SAMPLE_RATE // MODEL_RATE
    usable_samples = len(data) // factor * factor
    values = data[["HR (bpm)", "CIN2_pF", "CIN4_pF"]].to_numpy(np.float64)[:usable_samples]
    values = resample_poly(values, 1, factor, axis=0).astype(np.float32)
    labels = data["stage"].map(STAGE_TO_INDEX).to_numpy()[:usable_samples:factor][:len(values)]
    window = CONTEXT_EPOCHS * SAMPLES_PER_EPOCH
    examples, targets = [], []
    for start in range(0, len(values) - window + 1, stride_seconds * MODEL_RATE):
        center = start + CONTEXT_BEFORE * SAMPLES_PER_EPOCH
        examples.append(values[start:start + window])
        targets.append(int(np.bincount(labels[center:center + SAMPLES_PER_EPOCH], minlength=len(STAGES)).argmax()))
    return np.asarray(examples, dtype=np.float32), np.asarray(targets, dtype=np.int64)


def _standardize(train_x: np.ndarray, other_x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = train_x.reshape(-1, train_x.shape[-1]).mean(axis=0)
    std = train_x.reshape(-1, train_x.shape[-1]).std(axis=0)
    std[std < 1e-6] = 1.0
    return (train_x - mean) / std, (other_x - mean) / std


def _prepare_fold(validation_date: str) -> tuple[DataLoader, DataLoader, torch.Tensor]:
    """Train on every other sleep, validate on `validation_date`."""
    train_parts = [_make_examples(sleep_data[d], TRAIN_STRIDE_SECONDS) for d in recording_dates if d != validation_date]
    validation_x, validation_y = _make_examples(sleep_data[validation_date])
    train_x = np.concatenate([part[0] for part in train_parts])
    train_y = np.concatenate([part[1] for part in train_parts])
    train_x, validation_x = _standardize(train_x, validation_x)
    train_x = torch.from_numpy(train_x.transpose(0, 2, 1))
    validation_x = torch.from_numpy(validation_x.transpose(0, 2, 1))
    train_y = torch.from_numpy(train_y)
    validation_y = torch.from_numpy(validation_y)
    print("Train:", tuple(train_x.shape), "| Validation:", tuple(validation_x.shape))
    train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
    validation_loader = DataLoader(TensorDataset(validation_x, validation_y), batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
    return train_loader, validation_loader, train_y

In [10]:
class FastMaxPool1d(nn.Module):
    """Same result as nn.MaxPool1d(2) (an odd last sample is dropped), but much faster backward pass on the GPU."""

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, channels, length = x.shape
        return x[..., : length // 2 * 2].reshape(batch, channels, length // 2, 2).amax(-1)


def _autocast() -> torch.autocast:
    return torch.autocast("cuda", dtype=AUTOCAST_DTYPE or torch.float32, enabled=AUTOCAST_DTYPE is not None)


class EpochContextBranch(nn.Module):
    """Shared per-epoch encoder followed by a convolution across the 10 context epochs."""

    def __init__(self, channels: int, width: int = ENCODER_WIDTH) -> None:
        super().__init__()
        blocks = []
        in_channels = channels
        for out_channels in (width, width, 2 * width, 2 * width):
            blocks.extend([
                nn.Conv1d(in_channels, out_channels, kernel_size=7, padding=3, bias=False),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
                FastMaxPool1d(),
            ])
            in_channels = out_channels
        # Shared encoder: one 30-second epoch -> one embedding vector.
        self.epoch_encoder = nn.Sequential(*blocks, nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(DROPOUT))
        # Mixes neighbouring epoch embeddings across the 10-epoch context.
        self.context = nn.Sequential(
            nn.Conv1d(in_channels, in_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(in_channels),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
        )
        self.out_features = 2 * in_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, channels, _ = x.shape
        epochs = x.view(batch, channels, CONTEXT_EPOCHS, SAMPLES_PER_EPOCH).permute(0, 2, 1, 3)
        embeddings = self.epoch_encoder(epochs.reshape(batch * CONTEXT_EPOCHS, channels, SAMPLES_PER_EPOCH))
        context = self.context(embeddings.view(batch, CONTEXT_EPOCHS, -1).transpose(1, 2))
        return torch.cat([context[:, :, CONTEXT_BEFORE], context.mean(2)], 1)


def _initialize(module: nn.Module) -> None:
    if isinstance(module, (nn.Conv1d, nn.Linear)):
        nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
        if module.bias is not None:
            nn.init.zeros_(module.bias)


class SleepCNN(nn.Module):
    """Joint model: HR, CIN2 and CIN4 enter one branch together."""

    def __init__(self, channels: int = 3, classes: int = 4) -> None:
        super().__init__()
        self.branch = EpochContextBranch(channels)
        self.classifier = nn.Linear(self.branch.out_features, classes)
        self.apply(_initialize)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.branch(x))


class StagedSleepCNN(nn.Module):
    """HR branch is trained first; the CIN branch is added later through a zero-initialized residual head,
    so stage 2 starts exactly at the HR-only model."""

    def __init__(self, cin_channels: int = 2, classes: int = 4) -> None:
        super().__init__()
        self.hr = EpochContextBranch(1)
        self.hr_head = nn.Linear(self.hr.out_features, classes)
        self.cin = EpochContextBranch(cin_channels)
        self.fusion_head = nn.Linear(self.hr.out_features + self.cin.out_features, classes)
        self.use_cin = False
        self.apply(_initialize)
        nn.init.zeros_(self.fusion_head.weight)
        nn.init.zeros_(self.fusion_head.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hr_features = self.hr(x[:, :1])  # channel 0 is HR
        logits = self.hr_head(hr_features)
        if self.use_cin:
            logits = logits + self.fusion_head(torch.cat([hr_features, self.cin(x[:, 1:])], 1))
        return logits


def _build_model() -> nn.Module:
    return (StagedSleepCNN() if TRAINING_MODE == "staged" else SleepCNN()).to(DEVICE)


def _train_loss_function(train_y: torch.Tensor) -> nn.Module:
    class_counts = torch.bincount(train_y, minlength=len(STAGES)).float().clamp(min=1)
    class_weights = (class_counts.sum() / class_counts) ** CLASS_WEIGHT_POWER
    class_weights = (class_weights / class_weights.mean()).to(DEVICE)
    return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)


loss_function = nn.CrossEntropyLoss()  # unweighted, used for validation / early stopping
example_model = _build_model()
print("Training mode:", TRAINING_MODE)
print(example_model)
print("Parameters:", sum(p.numel() for p in example_model.parameters()))

Training mode: staged
StagedSleepCNN(
  (hr): EpochContextBranch(
    (epoch_encoder): Sequential(
      (0): Conv1d(1, 32, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
      (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): FastMaxPool1d()
      (4): Conv1d(32, 32, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
      (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): FastMaxPool1d()
      (8): Conv1d(32, 64, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
      (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (10): ReLU()
      (11): FastMaxPool1d()
      (12): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
      (13): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (14): ReLU()
      (15): FastMaxPool1d()
      (16): Adaptive

In [11]:
def _evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, np.ndarray]:
    model.eval()
    total_loss, predictions, targets = 0.0, [], []
    with torch.no_grad():
        for features, labels in loader:
            features = features.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with _autocast():
                logits = model(features)
            logits = logits.float()
            total_loss += loss_function(logits, labels).item() * labels.size(0)
            predictions.append(logits.argmax(1).cpu().numpy())
            targets.append(labels.cpu().numpy())
    return total_loss / len(loader.dataset), np.column_stack((np.concatenate(targets), np.concatenate(predictions)))


def _fit(
    model: nn.Module,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    train_loss_function: nn.Module,
    parameter_groups,
    label: str,
    frozen: tuple[nn.Module, ...] = (),
) -> float:
    """Train until early stopping; the starting state counts as a candidate, so a stage that does not
    improve validation loss leaves the model unchanged."""
    optimizer = torch.optim.AdamW(parameter_groups, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=LR_PATIENCE)
    best_loss, _ = _evaluate(model, validation_loader)
    best_state, stale = copy.deepcopy(model.state_dict()), 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for module in frozen:  # keep frozen BatchNorm statistics fixed
            module.eval()
        for features, labels in train_loader:
            features = features.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with _autocast():
                loss = train_loss_function(model(features), labels)
            loss.backward()
            optimizer.step()
        validation_loss, _ = _evaluate(model, validation_loader)
        scheduler.step(validation_loss)
        if validation_loss < best_loss - 1e-6:
            best_loss, best_state, stale = validation_loss, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
        if epoch == 1 or epoch % 10 == 0:
            print(f"[{label}] epoch={epoch:03d} validation_loss={validation_loss:.5f} lr={optimizer.param_groups[0]['lr']:.2e}")
        if stale >= EARLY_STOPPING_PATIENCE:
            print(f"[{label}] early stopping at epoch", epoch)
            break
    model.load_state_dict(best_state)
    print(f"[{label}] best validation loss:", best_loss)
    return best_loss


def _train(model: nn.Module, train_loader: DataLoader, validation_loader: DataLoader, train_y: torch.Tensor) -> None:
    train_loss_function = _train_loss_function(train_y)
    if TRAINING_MODE == "staged":
        hr_parameters = [*model.hr.parameters(), *model.hr_head.parameters()]
        _fit(model, train_loader, validation_loader, train_loss_function, hr_parameters, "stage 1: HR")
        model.use_cin = True
        for parameter in hr_parameters:
            parameter.requires_grad_(HR_FINETUNE_LR > 0)
        parameter_groups = [{"params": [*model.cin.parameters(), *model.fusion_head.parameters()]}]
        if HR_FINETUNE_LR > 0:
            parameter_groups.append({"params": hr_parameters, "lr": HR_FINETUNE_LR})
        frozen = () if HR_FINETUNE_LR > 0 else (model.hr,)
        _fit(model, train_loader, validation_loader, train_loss_function, parameter_groups, "stage 2: HR + CIN", frozen)
    else:
        _fit(model, train_loader, validation_loader, train_loss_function, model.parameters(), "joint")

In [12]:
def classification_report(
    targets: np.ndarray, predictions: np.ndarray, validation_loss: float, validation_date: str
) -> tuple[pd.DataFrame, pd.Series]:
    matrix = np.zeros((len(STAGES), len(STAGES)), dtype=np.int64)
    for target, prediction in zip(targets, predictions):
        matrix[target, prediction] += 1
    rows = []
    for index, stage in enumerate(STAGES):
        tp = matrix[index, index]
        precision = tp / matrix[:, index].sum() if matrix[:, index].sum() else 0.0
        recall = tp / matrix[index].sum() if matrix[index].sum() else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({"stage": stage, "precision": precision, "recall": recall, "f1": f1, "support": matrix[index].sum()})
    report = pd.DataFrame(rows).set_index("stage")
    # Stages absent from the validation night are excluded from macro averages.
    present = report[report["support"] > 0]
    macro = present[["precision", "recall", "f1"]].mean()
    weighted = present[["precision", "recall", "f1"]].mul(present["support"], axis=0).sum() / present["support"].sum()
    report.loc["macro avg"] = [*macro, present["support"].sum()]
    report.loc["weighted avg"] = [*weighted, present["support"].sum()]
    report["support"] = report["support"].astype(int)

    total = matrix.sum()
    accuracy = np.trace(matrix) / total if total else 0.0
    expected = (matrix.sum(axis=0) * matrix.sum(axis=1)).sum() / total**2 if total else 0.0
    kappa = (accuracy - expected) / (1 - expected) if expected < 1 else 0.0
    awake = STAGE_TO_INDEX["Awake"]
    true_sleep, predicted_sleep = targets != awake, predictions != awake
    sleep_wake_accuracy = (true_sleep == predicted_sleep).mean()
    tst_true = true_sleep.sum() * EPOCH_SECONDS / 3600
    tst_predicted = predicted_sleep.sum() * EPOCH_SECONDS / 3600
    summary = pd.Series({
        "Validation loss": validation_loss,
        "Accuracy": accuracy,
        "Balanced accuracy (macro recall)": macro["recall"],
        "Macro F1": macro["f1"],
        "Weighted F1": weighted["f1"],
        "Cohen kappa": kappa,
        "Sleep/wake accuracy": sleep_wake_accuracy,
        "Sleep sensitivity": predicted_sleep[true_sleep].mean() if true_sleep.any() else np.nan,
        "Wake specificity": (~predicted_sleep[~true_sleep]).mean() if (~true_sleep).any() else np.nan,
        "TST ground truth (h)": tst_true,
        "TST CNN (h)": tst_predicted,
        "TST error (min)": (tst_predicted - tst_true) * 60,
    })
    print("Validation sleep:", validation_date, "|", total, "epochs")
    display(summary.round(4).to_frame("value"))
    print("Confusion matrix (rows=true, columns=predicted):")
    display(pd.DataFrame(matrix, index=STAGES, columns=STAGES))
    print("Stage durations (minutes):")
    display(pd.DataFrame({
        "true": matrix.sum(axis=1) * EPOCH_SECONDS / 60,
        "predicted": matrix.sum(axis=0) * EPOCH_SECONDS / 60,
    }, index=STAGES))
    display(report.round(3))
    return report, summary


def run_fold(validation_date: str) -> tuple[nn.Module, pd.DataFrame, pd.Series]:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    print(f"\n========== Validation sleep: {validation_date} ==========")
    train_loader, validation_loader, train_y = _prepare_fold(validation_date)
    model = _build_model()
    _train(model, train_loader, validation_loader, train_y)
    validation_loss, result = _evaluate(model, validation_loader)
    report, summary = classification_report(result[:, 0], result[:, 1], validation_loss, validation_date)
    return model, report, summary


validation_dates = recording_dates if CROSS_VALIDATE else [VALIDATION_DATE]
fold_results = {validation_date: run_fold(validation_date) for validation_date in validation_dates}
model, report, summary = fold_results[validation_dates[-1]]  # last fold, kept for further inspection

if CROSS_VALIDATE:
    summaries = pd.DataFrame({date: fold[2] for date, fold in fold_results.items()}).T
    stage_f1 = pd.DataFrame({
        date: fold[1]["f1"].where(fold[1]["support"] > 0)  # absent stages -> NaN, skipped by mean
        for date, fold in fold_results.items()
    }).T
    for table in (summaries, stage_f1):
        mean, std = table.mean(), table.std()
        table.loc["mean"], table.loc["std"] = mean, std
    print(f"\n========== Leave-one-night-out summary ({len(validation_dates)} folds) ==========")
    display(summaries.round(3))
    print("Per-stage F1 by validation night:")
    display(stage_f1.round(3))


========== Validation sleep: 2026-10-16 ==========
Train: (4340, 3, 600) | Validation: (729, 3, 600)
[stage 1: HR] epoch=001 validation_loss=1.35871 lr=1.00e-03
[stage 1: HR] epoch=010 validation_loss=1.36669 lr=1.00e-03
[stage 1: HR] epoch=020 validation_loss=1.71455 lr=1.00e-03
[stage 1: HR] epoch=030 validation_loss=1.22128 lr=5.00e-04
[stage 1: HR] epoch=040 validation_loss=1.32548 lr=2.50e-04
[stage 1: HR] early stopping at epoch 41
[stage 1: HR] best validation loss: 1.1969556925391627
[stage 2: HR + CIN] epoch=001 validation_loss=1.45754 lr=1.00e-03
[stage 2: HR + CIN] epoch=010 validation_loss=1.12068 lr=1.00e-03
[stage 2: HR + CIN] epoch=020 validation_loss=1.26471 lr=1.00e-03
[stage 2: HR + CIN] epoch=030 validation_loss=1.04495 lr=5.00e-04
[stage 2: HR + CIN] epoch=040 validation_loss=1.00013 lr=5.00e-04
[stage 2: HR + CIN] epoch=050 validation_loss=1.05098 lr=5.00e-04
[stage 2: HR + CIN] epoch=060 validation_loss=1.01704 lr=2.50e-04
[stage 2: HR + CIN] epoch=070 validation

,value
Validation loss,0.9698
Accuracy,0.6626
Balanced accuracy (macro recall),0.4893
Macro F1,0.5058
Weighted F1,0.6369
Cohen kappa,0.4211
Sleep/wake accuracy,0.9643
Sleep sensitivity,0.9957
Wake specificity,0.1786
TST ground truth (h),5.8417


Confusion matrix (rows=true, columns=predicted):


,Awake,REM,Core,Deep
Awake,5,3,20,0
REM,1,135,32,11
Core,2,48,314,27
Deep,0,17,85,29


Stage durations (minutes):


...

,precision,recall,f1,support
stage,,,,
Awake,0.000,0.000,0.000,11
REM,0.284,0.778,0.416,27
Core,0.788,0.603,0.684,179
Deep,0.584,0.591,0.588,88
macro avg,0.414,0.493,0.422,305
weighted avg,0.656,0.593,0.608,305



========== Leave-one-night-out summary (4 folds) ==========


,Validation loss,Accuracy,Balanced accuracy (macro recall),Macro F1,Weighted F1,Cohen kappa,Sleep/wake accuracy,Sleep sensitivity,Wake specificity,TST ground truth (h),TST CNN (h),TST error (min)
2026-10-16,0.970,0.663,0.489,0.506,0.637,0.421,0.964,0.996,0.179,5.842,6.008,10.000
2026-10-17,0.779,0.700,0.684,0.672,0.698,0.559,0.931,0.950,0.675,4.633,4.508,-7.500
2026-10-19,0.681,0.757,0.669,0.682,0.755,0.635,0.971,0.991,0.353,4.417,4.467,3.000
2026-10-20,1.194,0.593,0.493,0.422,0.608,0.355,0.948,0.983,0.000,2.450,2.500,3.000
mean,0.906,0.678,0.584,0.570,0.674,0.492,0.953,0.980,0.302,4.335,4.371,2.125
std,0.227,0.068,0.107,0.128,0.066,0.128,0.018,0.021,0.288,1.405,1.439,7.215


Per-stage F1 by validation night:


stage,Awake,REM,Core,Deep,macro avg,weighted avg
2026-10-16,0.278,0.707,0.746,0.293,0.506,0.637
2026-10-17,0.568,0.661,0.718,0.740,0.672,0.698
2026-10-19,0.429,0.800,0.759,0.740,0.682,0.755
2026-10-20,0.000,0.416,0.684,0.588,0.422,0.608
mean,0.319,0.646,0.727,0.590,0.570,0.674
std,0.243,0.164,0.033,0.211,0.128,0.066
